# Shape-fix test: conditional sigma_gap for long-gap targets

**Date:** 2026-04-18.
**Question:** If we use σ_gap=4 (tight) for targets with gap > 10d, but keep σ_gap=8 for shorter-gap targets, does it help the long-gap phase-1 under-prediction without hurting the rest of the cohort?

**Caveat known in advance:** only 2 of the 5 h/m under-predicted movies are long-gap (`forbidden_fruits` 13.58d, `they_will_kill_you` 12.58d). The other 3 are short-gap high-volume (`the_drama`, `super_mario_galaxy`, `you_me_and_tuscany`). This test can only rule in/out the narrow long-gap hypothesis; short-gap high-volume remains a Path B target.

**Test:**
- Ship config (control): σ_gap=8 for all targets.
- Candidate: σ_gap=4 if `target_gap > 10d`, else σ_gap=8.
- Ship stack otherwise (combined_score α=0.5, ceil=0.7d, n=20, scaling ship-defaults).
- Measure phase-1 MAE at T-3d on (a) full cohort, (b) long-gap subset (target_gap > 10d), (c) 5 h/m targets.

**Decision rule:** candidate is viable if it reduces long-gap subset MAE without worsening the remaining-cohort MAE.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from _helpers import (
    reviews, close_date_map, gap_lookup, first_review_ts, gap_for_slug,
    combined_score_selector,
    snapshot_state, close_day_count,
    build_critic_profiles, build_kde_lambda_model_capped,
    predict_window_custom,
    passes_skip_rules_for_snap,
    CACHE_DIR,
)

SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP_DEFAULT = 8.0
SHIP_N_TRAINING = 20
SHIP_BANDWIDTH_FLOOR = 0.5
SHIP_BANDWIDTH_CEIL = 0.7
SNAP = 3.0

LONG_GAP_THRESHOLD = 10.0
LONG_GAP_SIGMA = 4.0

CACHE = CACHE_DIR / 'shape_fix_long_gap.pkl'
print('Ready. Long-gap sigma_gap = 4 when target_gap > 10d; else 8 (ship default).')

## Run both configurations at T-3d

In [ ]:
def run_phase1_for_sigma(sigma_gap_fn, force=False):
    """Run phase-1 prediction across the cohort at T-3d with a sigma_gap picked per-target.

    sigma_gap_fn: callable(target_slug, target_gap) -> sigma_gap
    """
    rows = []
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
        snap_time = target_close - pd.Timedelta(days=SNAP)
        state = snapshot_state(target, snap_time)
        passed, _ = passes_skip_rules_for_snap(state, SNAP)
        if not passed:
            continue

        target_window_days = state['first_review_dbc'] - SNAP
        target_critics = state['observed_critics']

        sigma_gap = sigma_gap_fn(target, target_gap)
        training, _ = combined_score_selector(
            target, target_gap, target_critics, target_window_days,
            k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=sigma_gap,
        )
        if len(training) < 5:
            continue

        profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
        model = build_kde_lambda_model_capped(
            profiles,
            bandwidth_floor=SHIP_BANDWIDTH_FLOOR,
            bandwidth_ceiling=SHIP_BANDWIDTH_CEIL,
        )
        phase1 = predict_window_custom(
            model, dbc_from=SNAP, dbc_to=midnight_utc_dbc,
            observed_critics=target_critics,
            observed_count=state['observed_count'],
            first_review_dbc=state['first_review_dbc'],
        )

        movie_reviews = reviews[reviews['movie_slug'] == target].copy()
        movie_reviews['dbc'] = (target_close - movie_reviews['estimated_timestamp']).dt.total_seconds() / 86400
        actual_p1 = int(((movie_reviews['dbc'] > midnight_utc_dbc) & (movie_reviews['dbc'] <= SNAP)).sum())

        rows.append({
            'target': target,
            'target_gap': target_gap,
            'sigma_gap_used': sigma_gap,
            'phase1_pred': float(phase1),
            'actual_phase1': actual_p1,
            'err': float(phase1) - actual_p1,
            'abs_err': abs(float(phase1) - actual_p1),
        })
    return pd.DataFrame(rows)


def sigma_control(_target, _target_gap):
    return SHIP_SIGMA_GAP_DEFAULT


def sigma_candidate(_target, target_gap):
    return LONG_GAP_SIGMA if target_gap > LONG_GAP_THRESHOLD else SHIP_SIGMA_GAP_DEFAULT


if CACHE.exists():
    cached = pd.read_pickle(CACHE)
    control = cached[cached['config'] == 'control'].copy()
    candidate = cached[cached['config'] == 'candidate'].copy()
    print(f'Loaded from cache: control {len(control)}, candidate {len(candidate)}')
else:
    print('Running control (sigma_gap=8 everywhere)...')
    control = run_phase1_for_sigma(sigma_control)
    control['config'] = 'control'
    print(f'  kept {len(control)}')

    print('Running candidate (sigma_gap=4 if gap>10, else 8)...')
    candidate = run_phase1_for_sigma(sigma_candidate)
    candidate['config'] = 'candidate'
    print(f'  kept {len(candidate)}')

    combined_cache = pd.concat([control, candidate], ignore_index=True)
    combined_cache.to_pickle(CACHE)
    print(f'Cached {len(combined_cache)} rows')

print()
print('Candidate sigma_gap distribution:')
print(candidate['sigma_gap_used'].value_counts().to_string())

## Cohort-wide comparison

In [ ]:
joined = control.merge(
    candidate[['target', 'phase1_pred', 'err', 'abs_err']],
    on='target', suffixes=('_ctrl', '_cand'),
)
joined['delta_abs_err'] = joined['abs_err_ctrl'] - joined['abs_err_cand']  # positive = candidate better

def summarize(df, label):
    print(f'{label:30s}  n={len(df):3d}  ctrl_MAE={df["abs_err_ctrl"].mean():6.2f}  '
          f'cand_MAE={df["abs_err_cand"].mean():6.2f}  '
          f'delta={df["delta_abs_err"].mean():+6.2f}  '
          f'delta_pct={100*df["delta_abs_err"].mean()/df["abs_err_ctrl"].mean():+6.1f}%')

print('MAE comparison (positive delta = candidate better):')
print()
summarize(joined, 'Full cohort')
print()

long = joined[joined['target_gap'] > LONG_GAP_THRESHOLD]
short = joined[joined['target_gap'] <= LONG_GAP_THRESHOLD]
summarize(long, 'Long-gap subset (gap>10d)')
summarize(short, 'Short-gap subset (gap<=10d)')
print()

HM_TARGETS = ['the_drama', 'the_super_mario_galaxy_movie',
              'forbidden_fruits_2026', 'they_will_kill_you', 'you_me_and_tuscany']
hm = joined[joined['target'].isin(HM_TARGETS)]
summarize(hm, 'H/m subset (5 movies)')

hm_long = hm[hm['target_gap'] > LONG_GAP_THRESHOLD]
summarize(hm_long, 'H/m long-gap subset (2 movies)')

## Per-target inspection for the affected movies

In [ ]:
affected = joined[joined['target_gap'] > LONG_GAP_THRESHOLD].sort_values('target_gap', ascending=False)
print(f'All long-gap targets (n={len(affected)}):')
cols = ['target', 'target_gap', 'actual_phase1', 'phase1_pred_ctrl', 'phase1_pred_cand',
        'err_ctrl', 'err_cand', 'delta_abs_err']
print(affected[cols].head(30).to_string(index=False, float_format='%.2f'))
print()
print('Long-gap mean_err:')
print(f'  control:    {affected["err_ctrl"].mean():+.2f}')
print(f'  candidate:  {affected["err_cand"].mean():+.2f}')

## Decision

Compare long-gap MAE improvement vs short-gap MAE preservation:
- If long-gap MAE meaningfully improves (say ≥10%) AND short-gap MAE non-worse → candidate is a viable targeted fix.
- If long-gap doesn't improve or short-gap regresses → gap-narrowing isn't the answer. Concede: the under-prediction is driven by volume (the_drama at gap=6.37 is the biggest miss), which is Path B territory.